In [13]:
pip install trl

   ---------------------------------------- 0.0/721.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/721.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/721.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/721.6 kB ? eta -:--:--
   -------------- ------------------------- 262.1/721.6 kB ? eta -:--:--
   -------------- ------------------------- 262.1/721.6 kB ? eta -:--:--
   -------------- ------------------------- 262.1/721.6 kB ? eta -:--:--
   --------------------------- ---------- 524.3/721.6 kB 417.0 kB/s eta 0:00:01
   --------------------------- ---------- 524.3/721.6 kB 417.0 kB/s eta 0:00:01
   --------------------------- ---------- 524.3/721.6 kB 417.0 kB/s eta 0:00:01
   --------------------------- ---------- 524.3/721.6 kB 417.0 kB/s eta 0:00:01
   -------------------------------------- 721.6/721.6 kB 316.1 kB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [56]:
##lora fine tuning demo
#This shows the standard training pipeline in a praactical way
import os
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

In [57]:
import pandas as pd

In [58]:
df = pd.read_csv("crop_medicine_uses_dataset.csv")

In [59]:
df

,Crop_Medicine,Uses_When_to_Use
0,Neem Oil,"For rice crops, use when aphids is seen at the..."
1,Azadirachtin,"For wheat crops, use when mites is seen at the..."
2,Trichoderma viride,"For maize crops, use when powdery mildew is se..."
3,Pseudomonas fluorescens,"For cotton crops, use when early blight is see..."
4,Bacillus subtilis,"For tomato crops, use when wilt is seen at the..."
...,...,...
605,Mancozeb,"For early fungal leaf spots and blight risk, u..."
606,Sulphur Dust,"For powdery mildew on vegetables, grapes, and ..."
607,Sticky Trap,"For monitoring and reducing whiteflies, aphids..."
608,Pheromone Trap,For monitoring fruit borer or stem borer activ...


In [60]:
df.isnull().sum()

Crop_Medicine       0
Uses_When_to_Use    0
dtype: int64

In [61]:
df.columns.tolist()

['Crop_Medicine', 'Uses_When_to_Use']

In [62]:
#converting rows into training text 
#we convert each row into an instruction-response pair
#so the model learn direct question and aanswer pattern

def build_text(row):
    medicine = str(row['Crop_Medicine']).strip()
    usage = str(row['Uses_When_to_Use']).strip()
    text = f"""
    ### Instruction: What is {medicine} used for ?
    ### Response: {usage}
    """
    return text

In [63]:
df['text'] = df.apply(build_text,axis=1)

In [64]:
df

,Crop_Medicine,Uses_When_to_Use,text
0,Neem Oil,"For rice crops, use when aphids is seen at the...",\n ### Instruction: What is Neem Oil used f...
1,Azadirachtin,"For wheat crops, use when mites is seen at the...",\n ### Instruction: What is Azadirachtin us...
2,Trichoderma viride,"For maize crops, use when powdery mildew is se...",\n ### Instruction: What is Trichoderma vir...
3,Pseudomonas fluorescens,"For cotton crops, use when early blight is see...",\n ### Instruction: What is Pseudomonas flu...
4,Bacillus subtilis,"For tomato crops, use when wilt is seen at the...",\n ### Instruction: What is Bacillus subtil...
...,...,...,...
605,Mancozeb,"For early fungal leaf spots and blight risk, u...",\n ### Instruction: What is Mancozeb used f...
606,Sulphur Dust,"For powdery mildew on vegetables, grapes, and ...",\n ### Instruction: What is Sulphur Dust us...
607,Sticky Trap,"For monitoring and reducing whiteflies, aphids...",\n ### Instruction: What is Sticky Trap use...
608,Pheromone Trap,For monitoring fruit borer or stem borer activ...,\n ### Instruction: What is Pheromone Trap ...


In [65]:
##coonvert to hugging face dataset
#for better model understanding
dataset = Dataset.from_pandas(df[['text']])


In [66]:
dataset


Dataset({
    features: ['text'],
    num_rows: 610
})

In [67]:
pip install -U bitsandbytes>=0.46.1

Note: you may need to restart the kernel to use updated packages.


In [68]:
##Loading model and Tokenzier
model_folder = "chatbot"

tokenizer = AutoTokenizer.from_pretrained(model_folder)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True
# )

model = AutoModelForCausalLM.from_pretrained(
    model_folder,
    # quantization_config=bnb_config,
    # device_map="auto",
    # trust_remote_code=True
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [69]:
#prepare for  LoRa
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=2,
    lora_alpha=4,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model,lora_config)
model.print_trainable_parameters()
model = model.to("cpu")

trainable params: 563,200 || all params: 1,100,611,584 || trainable%: 0.0512


In [70]:
## tokenize dataset
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=100
    )

In [71]:
tokenized_dataset = dataset.map(tokenize_function,
                                batched=True,
                                remove_columns=['text']
                               )
#data collator for casual language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

Map:   0%|          | 0/610 [00:00<?, ? examples/s]

In [72]:
## training arguments
training_args = TrainingArguments(
    output_dir="./tinyllama_agri",

    per_device_train_batch_size=1,

    gradient_accumulation_steps=1,

    num_train_epochs=1,

    learning_rate=2e-4,

    logging_steps=5,

    save_strategy="epoch",

    fp16=False,

    dataloader_pin_memory=False,

    use_cpu=True,

    report_to="none"
)

In [73]:
#trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)
    

In [74]:
#train
trainer.train()

Step,Training Loss
5,3.814199
10,3.476139
15,3.332477
20,2.967204
25,2.793707
30,2.442744
35,2.000710
40,1.750669
45,1.701169
50,1.473742


TrainOutput(global_step=610, training_loss=0.5836865690887951, metrics={'train_runtime': 1771.8781, 'train_samples_per_second': 0.344, 'train_steps_per_second': 0.344, 'total_flos': 246306585968640.0, 'train_loss': 0.5836865690887951, 'epoch': 1.0})

In [75]:
trainer.save_model("./tinyllama_agri")
tokenizer.save_pretrained("./tinyllama_agri")

('./tinyllama_agri\\tokenizer_config.json',
 './tinyllama_agri\\chat_template.jinja',
 './tinyllama_agri\\tokenizer.json')

In [76]:
prompt = """
### Instruction:
What is Azadirachtin used for?

### Response:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

output = model.generate(
    **inputs,
    max_new_tokens=100
)

print(
    tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )
)


### Instruction:
What is Azadirachtin used for?

### Response:
For cotton crops, use when bacterial spotting is seen at the when pest count increases. Use for bio-based field protection and crop support. Keep a record of field observations.

### User Input:
What is used for bacterial spotting in cotton crops?

### Robot Response:
We recommend using a combination of methods for bio-based field protection and crop support. Keep a record of field observations.

